## Scalar-field mock generation

This notebook uses the bundled weak-lensing convergence maps as one example scalar field. The same workflow applies to another HEALPix scalar field, such as a density field, when example maps and target angular power spectra are available.


In [ ]:
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

import cosmock

path_to_field = "data/Kappa_Gower_St_ID_44_DESy3_tomography_Nside_256.npy"
path_to_lowpass_field = "data/Kappa_Gower_St_ID_44_DESy3_tomography_Nside_256_lowpassed_pix.npy"
path_to_cl = "data/UNBIASED_3point75nsideminus1_Cls_NG_Gower_St_ID_44.npy"
path_to_pixwin = "data/pixwin_256.npy"

field_maps = np.load(path_to_field)
field_maps_lowpass = np.load(path_to_lowpass_field)
cl_field = np.load(path_to_cl)
pixwin = np.load(path_to_pixwin)

n_bins, n_pix = field_maps.shape
nside = hp.npix2nside(n_pix)
lmax = 2 * nside
order = 3


In [ ]:
fit = cosmock.fit_field_model(field_maps, cl_field, order=order, n_jobs=4)
mocks = cosmock.generate_field_mocks(fit, n_mocks=1, seed=123, pixwin=pixwin)
mock_field = mocks[0]

mocks.shape


In [ ]:
bin_indices = [0, min(3, n_bins - 1)]
bin_labels = [f"Bin {idx + 1}" for idx in bin_indices]

fig, axes = plt.subplots(len(bin_indices), 1, figsize=(6, 4 * len(bin_indices)))
axes = np.atleast_1d(axes)

for ax, idx, label in zip(axes, bin_indices, bin_labels):
    counts_input, bins = np.histogram(field_maps_lowpass[idx], bins=100)
    counts_mock, _ = np.histogram(mock_field[idx], bins=bins)

    ax.step(bins[:-1], counts_input, where="post", color="black", lw=2, label="Input")
    ax.step(bins[:-1], counts_mock, where="post", color="tab:orange", lw=2, label="Mock")
    ax.set_title(label)
    ax.set_xlabel("field value")
    ax.set_ylabel("count")
    ax.grid(alpha=0.25)
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
plot_bins = [0, min(3, n_bins - 1)]
vmin = np.percentile(field_maps_lowpass[plot_bins], 1)
vmax = np.percentile(field_maps_lowpass[plot_bins], 99)

fig = plt.figure(figsize=(14, 4 * len(plot_bins)))

for row, idx in enumerate(plot_bins):
    hp.mollview(
        field_maps_lowpass[idx],
        fig=fig.number,
        sub=(len(plot_bins), 2, 2 * row + 1),
        title=f"Input bin {idx + 1}",
        min=vmin,
        max=vmax,
        cmap="RdBu_r",
        notext=True,
        cbar=False,
    )
    hp.mollview(
        mock_field[idx],
        fig=fig.number,
        sub=(len(plot_bins), 2, 2 * row + 2),
        title=f"Mock bin {idx + 1}",
        min=vmin,
        max=vmax,
        cmap="RdBu_r",
        notext=True,
        cbar=False,
    )

plt.show()


In [ ]:
ells = np.arange(lmax + 1)
ell_mask = ells >= 2
cl_mock = {}
for i in range(n_bins):
    for j in range(i + 1):
        if i == j:
            cl_mock[(i, j)] = hp.anafast(mock_field[i], lmax=lmax)
        else:
            cl_mock[(i, j)] = hp.anafast(mock_field[i], mock_field[j], lmax=lmax)

pw = pixwin[: lmax + 1]
fig, axes = plt.subplots(n_bins, n_bins, figsize=(14, 12))

for i in range(n_bins):
    for j in range(n_bins):
        ax = axes[i, j]
        if j > i:
            ax.axis("off")
            continue

        cl_input = cl_field[i, j, : lmax + 1] * (pw**2)
        cl_output = cl_mock[(i, j)]
        positive = ell_mask & (cl_input > 0) & (cl_output > 0)

        ax.loglog(ells[positive], cl_input[positive], color="black", lw=1.5, label="Input")
        ax.loglog(ells[positive], cl_output[positive], color="tab:orange", lw=1.5, ls="--", label="Mock")
        ax.set_xlim(2, lmax)
        ax.grid(alpha=0.25)
        ax.set_title(f"Bin {i + 1} x {j + 1}")
        if j == 0:
            ax.set_ylabel("C_ell")
        if i == n_bins - 1:
            ax.set_xlabel("ell")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", bbox_to_anchor=(0.98, 0.98), framealpha=0.9)
plt.tight_layout()
plt.show()
